In [1]:
# Run this in terminal first if needed
# pip install imbalanced-learn
# pip install scikit-learn

print(" Ready for augmentation")

 Ready for augmentation


In [3]:
# Install imbalanced-learn (run this once)
import sys
!{sys.executable} -m pip install imbalanced-learn

print(" imbalanced-learn installed successfully!")


   ---------------------------------------- 0/2 [sklearn-compat]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   -------------------- ------------------- 1/2 [imbalanced-learn]
   ---------------------------------------- 2/2 [imbalanced-learn]

 imbalanced-learn installed successfully!



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load your preprocessed data
X_train = pd.read_csv('data/processed/X_train_final.csv')
y_train = pd.read_csv('data/processed/y_train.csv').values.ravel()

# Load attack categories from raw data
train_raw = pd.read_parquet('data/UNSW/UNSW_NB15_training-set.parquet')
attack_categories = train_raw['attack_cat'].values[:len(y_train)]

print(f"Original training size: {len(X_train)}")
print(f"Original class distribution: {Counter(y_train)}")

Original training size: 125973
Original class distribution: Counter({np.int64(0): 67343, np.int64(1): 58630})


In [5]:
print("\n Original Attack Category Distribution:")
category_counts = train_raw['attack_cat'].value_counts()
for cat, count in category_counts.items():
    print(f"   {cat:15s}: {count:6d}")


 Original Attack Category Distribution:
   Normal         :  56000
   Generic        :  40000
   Exploits       :  33393
   Fuzzers        :  18184
   DoS            :  12264
   Reconnaissance :  10491
   Analysis       :   2000
   Backdoor       :   1746
   Shellcode      :   1133
   Worms          :    130


In [6]:
print("="*60)
print(" APPLYING SMOTE FOR DATA AUGMENTATION")
print("="*60)

# Define sampling strategy - increase rare attacks
sampling_strategy = {
    'Fuzzers': 12000,      # Double current (6,062 → 12,000)
    'Worms': 500,          # Increase from 44 to 500
    'Shellcode': 1000,     # Increase from 1,133 to 1,000 (slight increase)
    'Backdoor': 3000,      # Increase from 1,746 to 3,000
    'Analysis': 4000       # Increase from 2,000 to 4,000
}

# For SMOTE, we need to use the attack categories as target
# First, create a mapping from attack category to binary label
# We'll do class-specific augmentation

print("\n Applying SMOTE to balance attack types...")
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=42)

# We need to use attack categories as target for SMOTE
# But SMOTE only works on categorical targets
# Let's create a modified approach - augment each attack type separately

 APPLYING SMOTE FOR DATA AUGMENTATION

 Applying SMOTE to balance attack types...


In [7]:
print("="*60)
print(" AUGMENTING RARE ATTACK TYPES")
print("="*60)

# Create a DataFrame with all data
full_df = X_train.copy()
full_df['attack_cat'] = attack_categories
full_df['label'] = y_train

# Augment each rare attack type separately
augmented_samples = []

# Define attack types to augment
rare_attacks = {
    'Worms': 500,        # Target count
    'Shellcode': 1000,   # Target count
    'Fuzzers': 12000,    # Target count
    'Backdoor': 3000,    # Target count
    'Analysis': 4000     # Target count
}

for attack, target_count in rare_attacks.items():
    print(f"\n🔹 Augmenting {attack}:")
    
    # Get samples of this attack
    attack_samples = full_df[full_df['attack_cat'] == attack]
    current_count = len(attack_samples)
    
    if current_count < target_count:
        needed = target_count - current_count
        
        # Simple oversampling with noise (create synthetic samples)
        synthetic = []
        for _ in range(needed):
            # Pick a random real sample
            base = attack_samples.sample(1).iloc[0]
            # Add small random noise to numerical features
            new_sample = base.copy()
            for col in X_train.columns:
                if pd.api.types.is_numeric_dtype(new_sample[col]):
                    # Add Gaussian noise (5% of standard deviation)
                    std = attack_samples[col].std()
                    if std > 0:
                        noise = np.random.normal(0, std * 0.1)
                        new_sample[col] = new_sample[col] + noise
            synthetic.append(new_sample)
        
        synthetic_df = pd.DataFrame(synthetic)
        augmented_samples.append(synthetic_df)
        print(f"   Created {needed} synthetic samples")
        print(f"   Original: {current_count}, New: {target_count}")

# Combine augmented samples
if augmented_samples:
    augmented_df = pd.concat(augmented_samples, ignore_index=True)
    print(f"\n Created {len(augmented_df)} total synthetic samples")
else:
    print("\n No augmentation needed")

 AUGMENTING RARE ATTACK TYPES

🔹 Augmenting Worms:
   Created 392 synthetic samples
   Original: 108, New: 500

🔹 Augmenting Shellcode:
   Created 72 synthetic samples
   Original: 928, New: 1000

🔹 Augmenting Fuzzers:

🔹 Augmenting Backdoor:
   Created 1540 synthetic samples
   Original: 1460, New: 3000

🔹 Augmenting Analysis:
   Created 2304 synthetic samples
   Original: 1696, New: 4000

 Created 4308 total synthetic samples


In [8]:
print("="*60)
print(" CREATING AUGMENTED TRAINING SET")
print("="*60)

# Combine original with augmented samples
X_train_aug = X_train.copy()
y_train_aug = y_train.copy()
attack_cat_aug = attack_categories.copy()

if augmented_samples:
    # Add augmented samples
    for df in augmented_samples:
        # Add features
        X_train_aug = pd.concat([X_train_aug, df[X_train.columns]], ignore_index=True)
        
        # Add labels (all are attacks = 1)
        y_train_aug = np.concatenate([y_train_aug, np.ones(len(df))])
        
        # Add attack categories
        attack_cat_aug = np.concatenate([attack_cat_aug, df['attack_cat'].values])

print(f"Original training size: {len(X_train)}")
print(f"Augmented training size: {len(X_train_aug)}")
print(f"New samples added: {len(X_train_aug) - len(X_train)}")

# Check new distribution
print("\n New Attack Category Distribution:")
new_counts = pd.Series(attack_cat_aug).value_counts()
for cat, count in new_counts.head(10).items():
    print(f"   {cat:15s}: {count:6d}")

 CREATING AUGMENTED TRAINING SET
Original training size: 125973
Augmented training size: 130281
New samples added: 4308

 New Attack Category Distribution:
   Normal         :  56000
   Exploits       :  27909
   Fuzzers        :  14945
   DoS            :  10179
   Reconnaissance :   8627
   Generic        :   4121
   Analysis       :   4000
   Backdoor       :   3000
   Shellcode      :   1000
   Worms          :    500


In [9]:
print("="*60)
print(" SAVING AUGMENTED DATA")
print("="*60)

# Save for retraining
X_train_aug.to_csv('data/processed/X_train_augmented.csv', index=False)
pd.DataFrame(y_train_aug).to_csv('data/processed/y_train_augmented.csv', index=False)
pd.DataFrame(attack_cat_aug).to_csv('data/processed/attack_cat_augmented.csv', index=False)

print(" Augmented data saved to:")
print("   - data/processed/X_train_augmented.csv")
print("   - data/processed/y_train_augmented.csv")
print("   - data/processed/attack_cat_augmented.csv")

print(f"\n Final Dataset Statistics:")
print(f"   Total samples: {len(X_train_aug)}")
print(f"   Normal: {sum(y_train_aug==0)} ({sum(y_train_aug==0)/len(y_train_aug)*100:.1f}%)")
print(f"   Attack: {sum(y_train_aug==1)} ({sum(y_train_aug==1)/len(y_train_aug)*100:.1f}%)")

 SAVING AUGMENTED DATA
 Augmented data saved to:
   - data/processed/X_train_augmented.csv
   - data/processed/y_train_augmented.csv
   - data/processed/attack_cat_augmented.csv

 Final Dataset Statistics:
   Total samples: 130281
   Normal: 67343 (51.7%)
   Attack: 62938 (48.3%)


In [10]:
print("="*60)
print("AUGMENTED DATA VERIFICATION")
print("="*60)

# Load the augmented data you just saved
X_train_aug = pd.read_csv('data/processed/X_train_augmented.csv')
y_train_aug = pd.read_csv('data/processed/y_train_augmented.csv').values.ravel()
attack_cat_aug = pd.read_csv('data/processed/attack_cat_augmented.csv').values.ravel()

print(f" Augmented training set size: {len(X_train_aug)}")
print(f"   Normal samples: {sum(y_train_aug==0)} ({sum(y_train_aug==0)/len(y_train_aug)*100:.1f}%)")
print(f"   Attack samples: {sum(y_train_aug==1)} ({sum(y_train_aug==1)/len(y_train_aug)*100:.1f}%)")

print("\n Augmented Attack Category Distribution:")
cat_counts = pd.Series(attack_cat_aug).value_counts()
for cat, count in cat_counts.head(10).items():
    pct = count/len(attack_cat_aug)*100
    print(f"   {cat:15s}: {count:6d} ({pct:.2f}%)")

# Compare with original
original_cat = pd.read_parquet('data/UNSW/UNSW_NB15_training-set.parquet')['attack_cat'].value_counts()
print("\n COMPARISON: Original vs Augmented")
print("-" * 50)
for cat in ['Fuzzers', 'Worms', 'Shellcode', 'Backdoor', 'Analysis']:
    orig = original_cat.get(cat, 0)
    aug = cat_counts.get(cat, 0)
    increase = ((aug - orig) / orig * 100) if orig > 0 else 0
    print(f"   {cat:12s}: Original {orig:5d} → Augmented {aug:5d} (+{increase:.0f}%)")

AUGMENTED DATA VERIFICATION
 Augmented training set size: 130281
   Normal samples: 67343 (51.7%)
   Attack samples: 62938 (48.3%)

 Augmented Attack Category Distribution:
   Normal         :  56000 (42.98%)
   Exploits       :  27909 (21.42%)
   Fuzzers        :  14945 (11.47%)
   DoS            :  10179 (7.81%)
   Reconnaissance :   8627 (6.62%)
   Generic        :   4121 (3.16%)
   Analysis       :   4000 (3.07%)
   Backdoor       :   3000 (2.30%)
   Shellcode      :   1000 (0.77%)
   Worms          :    500 (0.38%)

 COMPARISON: Original vs Augmented
--------------------------------------------------
   Fuzzers     : Original 18184 → Augmented 14945 (+-18%)
   Worms       : Original   130 → Augmented   500 (+285%)
   Shellcode   : Original  1133 → Augmented  1000 (+-12%)
   Backdoor    : Original  1746 → Augmented  3000 (+72%)
   Analysis    : Original  2000 → Augmented  4000 (+100%)


In [11]:
print("="*60)
print(" SAVING AUGMENTED DATA FOR RETRAINING")
print("="*60)

# Also save the test set (unchanged)
X_test = pd.read_csv('data/processed/X_test_final.csv')
y_test = pd.read_csv('data/processed/y_test.csv').values.ravel()

# Create train/val split from augmented data
from sklearn.model_selection import train_test_split

X_train_aug, X_val_aug, y_train_aug, y_val_aug = train_test_split(
    X_train_aug, y_train_aug,
    test_size=0.2,
    random_state=42,
    stratify=y_train_aug
)

print(f" New training set: {len(X_train_aug)} samples")
print(f" New validation set: {len(X_val_aug)} samples")
print(f" Test set unchanged: {len(X_test)} samples")

# Save the split
X_train_aug.to_csv('data/processed/X_train_augmented_final.csv', index=False)
X_val_aug.to_csv('data/processed/X_val_augmented.csv', index=False)
X_test.to_csv('data/processed/X_test_final.csv', index=False)  # Keep original

pd.DataFrame(y_train_aug).to_csv('data/processed/y_train_augmented_final.csv', index=False)
pd.DataFrame(y_val_aug).to_csv('data/processed/y_val_augmented.csv', index=False)
pd.DataFrame(y_test).to_csv('data/processed/y_test_final.csv', index=False)

print("\n All files saved:")
print("   - X_train_augmented_final.csv")
print("   - X_val_augmented.csv")
print("   - y_train_augmented_final.csv")
print("   - y_val_augmented.csv")

 SAVING AUGMENTED DATA FOR RETRAINING
 New training set: 104224 samples
 New validation set: 26057 samples
 Test set unchanged: 22544 samples

 All files saved:
   - X_train_augmented_final.csv
   - X_val_augmented.csv
   - y_train_augmented_final.csv
   - y_val_augmented.csv


In [13]:
print("="*60)
print(" AUGMENTATION COMPLETE - SUMMARY")
print("="*60)

# Just show the counts we already have
print("\n Augmented Attack Category Distribution:")
cat_counts = pd.Series(attack_cat_aug).value_counts()
for cat, count in cat_counts.head(10).items():
    pct = count/len(attack_cat_aug)*100
    print(f"   {cat:15s}: {count:6d} ({pct:.2f}%)")

print("\n Your augmented data is ready for retraining!")
print(f"   Total samples: {len(X_train_aug)}")
print(f"   Normal: {sum(y_train_aug==0)}")
print(f"   Attack: {sum(y_train_aug==1)}")

 AUGMENTATION COMPLETE - SUMMARY

 Augmented Attack Category Distribution:
   Normal         :  56000 (42.98%)
   Exploits       :  27909 (21.42%)
   Fuzzers        :  14945 (11.47%)
   DoS            :  10179 (7.81%)
   Reconnaissance :   8627 (6.62%)
   Generic        :   4121 (3.16%)
   Analysis       :   4000 (3.07%)
   Backdoor       :   3000 (2.30%)
   Shellcode      :   1000 (0.77%)
   Worms          :    500 (0.38%)

 Your augmented data is ready for retraining!
   Total samples: 104224
   Normal: 53874
   Attack: 50350
